# 🎭 Fine-tuning BERT avec Statistical Pooling
## Classification des émotions à partir de transcriptions texte

> Ce notebook fine-tune un modèle BERT pré-entraîné pour classer les émotions à partir des transcriptions Whisper.
> **Technique clé** : Statistical Pooling pour extraire les représentations contextuelles.

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
print(" Imports réussis!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
#  Charger la configuration
config_path = "../configs/text_model.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("\nConfiguration chargée:")
print(f"  - Modèle pré-entraîné: {config['model']['pretrained_model']}")
print(f"  - Nombre de classes: {config['model']['num_classes']}")
print(f"  - Max tokens: {config['tokenizer']['max_length']}")
print(f"  - Learning rate: {config['training']['learning_rate']}")
print(f"  - Batch size: {config['training']['batch_size']}")
print(f"  - Epochs: {config['training']['num_epochs']}")

# Paramètres
MODEL_NAME = config['model']['pretrained_model']
NUM_CLASSES = config['model']['num_classes']
MAX_LENGTH = config['tokenizer']['max_length']
BATCH_SIZE = config['training']['batch_size']
EPOCHS = config['training']['num_epochs']
LEARNING_RATE = config['training']['learning_rate']
WARMUP_STEPS = config['training']['warmup_steps']

# Mapping émotions
EMOTION_MAP = {
    'neutral': 0,
    'calm': 1,
    'happy': 2,
    'sad': 3,
    'angry': 4,
    'fearful': 5,
    'disgust': 6,
    'surprised': 7
}

REVERSE_EMOTION_MAP = {v: k for k, v in EMOTION_MAP.items()}

In [ ]:
#  Charger les données réelles de transcription
transcription_path = "../data/processed/transcriptions/full_dataset_with_text.csv"

# Charger le dataset
df = pd.read_csv(transcription_path)
print(f"✅ Dataset chargé: {len(df)} exemples")
print(f"Colonnes: {df.columns.tolist()}")
print(f"Émotions uniques: {df['emotion'].unique()}")
print(df.head())

# Mapper les émotions (harmoniser les noms)
emotion_mapping = {
    'fear': 'fearful',  # Mapper 'fear' vers 'fearful'
}
df['emotion'] = df['emotion'].replace(emotion_mapping)

# Garder seulement les colonnes nécessaires: text et emotion
train_data = df[['text', 'emotion']].copy()

# Créer les labels
train_data['label'] = train_data['emotion'].map(EMOTION_MAP)

# Vérifier qu'il n'y a pas d'émotions manquantes
missing_emotions = train_data[train_data['label'].isna()]['emotion'].unique()
if len(missing_emotions) > 0:
    print(f"⚠️ Émotions manquantes du mapping: {missing_emotions}")
    print("Elles seront ignorées...")
    train_data = train_data.dropna(subset=['label'])

print(f"\n✅ Données préparées: {len(train_data)} exemples")
print(f"Distribution:\n{train_data['emotion'].value_counts()}")

# Diviser en train/val/test (80% train, 10% val, 10% test)
from sklearn.model_selection import train_test_split

# Première split: 80% train, 20% temp (pour val + test)
train_df, temp_df = train_test_split(
    train_data, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_data['label']
)

# Deuxième split: diviser les 20% en 50/50 (10% val, 10% test)
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_df['label']
)

print(f"\n✅ Split effectué:")
print(f"  - Train: {len(train_df)} ({len(train_df)/len(train_data)*100:.1f}%)")
print(f"  - Val: {len(val_df)} ({len(val_df)/len(train_data)*100:.1f}%)")
print(f"  - Test: {len(test_df)} ({len(test_df)/len(train_data)*100:.1f}%)")

In [ ]:
#  Initialiser le tokenizer BERT
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer BertTokenizer chargé depuis {MODEL_NAME}")


# Créer le Dataset PyTorch
class TranscriptionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts.iloc[idx] if isinstance(self.texts, pd.Series) else self.texts[idx]
        label = self.labels[idx]
        
        # Tokenization
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


# Créer les DataLoaders
train_dataset = TranscriptionDataset(train_df['text'], train_df['label'].values, tokenizer)
val_dataset = TranscriptionDataset(val_df['text'], val_df['label'].values, tokenizer)
test_dataset = TranscriptionDataset(test_df['text'], test_df['label'].values, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f" DataLoaders créés:")
print(f"  - Train: {len(train_loader)} batches")
print(f"  - Val: {len(val_loader)} batches")
print(f"  - Test: {len(test_loader)} batches")

## 🎯 Architecture: BERT + Statistical Pooling

**Statistical Pooling** extrait une représentation globale du texte en faisant la moyenne des embeddings de TOUS les tokens:

$$\text{pooled} = \frac{1}{T} \sum_{t=1}^{T} h_t$$

où $h_t$ sont les embeddings des tokens et $T$ est le nombre de tokens.

**Avantages** par rapport au pooling par défaut:
- ✅ Capture l'information complète du texte
- ✅ Plus robuste que d'utiliser uniquement `[CLS]`
- ✅ Préserve les informations contextuelles

In [ ]:
class BertWithStatisticalPooling(nn.Module):
    """
    BERT avec Statistical Pooling pour la classification des émotions.
    
    Architecture:
    1. BERT (pré-entraîné) → contextualisé embeddings
    2. Statistical Pooling (moyenne sur tous les tokens)
    3. Couche de dropout (régularisation)
    4. Classification linéaire
    """
    
    def __init__(self, model_name, num_classes, dropout=0.1):
        super(BertWithStatisticalPooling, self).__init__()
        
        # Charger le modèle BERT pré-entraîné
        self.bert = BertModel.from_pretrained(model_name)
        self.bert_hidden_size = self.bert.config.hidden_size  # 768 pour bert-base
        
        # Couches de classification
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert_hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        """
        Forward pass.
        
        Args:
            input_ids: (batch_size, seq_length)
            attention_mask: (batch_size, seq_length)
            
        Returns:
            logits: (batch_size, num_classes)
        """
        # 1️⃣ BERT: obtenir les embeddings contextualisés
        # output shape: (batch_size, seq_length, 768)
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # (batch_size, seq_length, 768)
        
        # 2️⃣ Statistical Pooling: moyenne sur tous les tokens
        # Masquer les tokens de padding
        # attention_mask shape: (batch_size, seq_length)
        # Expand pour broadcasting: (batch_size, seq_length, 1)
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        
        # Sum avec masquage des tokens ignorés
        sum_hidden = (last_hidden_state * mask_expanded).sum(dim=1)  # (batch_size, 768)
        
        # Compter les tokens valides (non-padding)
        sum_mask = mask_expanded.sum(dim=1)  # (batch_size, 1)
        
        # Moyenne: sum / count
        pooled_output = sum_hidden / sum_mask  # (batch_size, 768)
        
        # 3️⃣ Dropout + Classification
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)  # (batch_size, num_classes)
        
        return logits


# Initialiser le modèle
model = BertWithStatisticalPooling(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,
    dropout=config['model']['hidden_dropout_prob']
)

model.to(device)
print(f"✅ Modèle créé et déplacé sur {device}")
print(f"   - Paramètres BERT: {sum(p.numel() for p in model.bert.parameters()):,}")
print(f"   - Paramètres totaux: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 🔧 Optimiseur et Scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=config['training']['weight_decay'])

# Nombre total de steps
total_steps = len(train_loader) * EPOCHS

# Scheduler linéaire avec warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

# Loss function
loss_function = nn.CrossEntropyLoss()

print(f"✅ Optimiseur et Scheduler configurés")
print(f"   - Total steps: {total_steps}")
print(f"   - Warmup steps: {WARMUP_STEPS}")

In [ ]:
def train_epoch(model, train_loader, optimizer, scheduler, loss_function, device):
    """Entraîner une époque."""
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    
    pbar = tqdm(train_loader, desc="Training")
    
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        logits = model(input_ids, attention_mask)
        
        # Compute loss
        loss = loss_function(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['training']['max_grad_norm'])
        optimizer.step()
        scheduler.step()
        
        # Metrics
        total_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_predictions += labels.size(0)
        
        pbar.set_postfix({'loss': total_loss / (pbar.n + 1)})
    
    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions
    
    return avg_loss, accuracy


def eval_epoch(model, val_loader, loss_function, device):
    """Évaluer sur une validation set."""
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validation")
        
        for batch in pbar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            logits = model(input_ids, attention_mask)
            
            # Compute loss
            loss = loss_function(logits, labels)
            
            # Metrics
            total_loss += loss.item()
            predictions = torch.argmax(logits, dim=1)
            correct_predictions += (predictions == labels).sum().item()
            total_predictions += labels.size(0)
            
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': total_loss / (pbar.n + 1)})
    
    avg_loss = total_loss / len(val_loader)
    accuracy = correct_predictions / total_predictions
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return avg_loss, accuracy, f1, all_preds, all_labels


print("✅ Fonctions d'entraînement et d'évaluation définies")

In [ ]:
# 🚀 Lancer l'entraînement
# ========================

# Créer le dossier pour les checkpoints
checkpoint_dir = "../checkpoints/bert_finetuned"
os.makedirs(checkpoint_dir, exist_ok=True)

# Historique des métriques
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': []
}

best_f1 = 0
best_epoch = 0
patience = 3
patience_counter = 0

print(f"\n📚 Début du fine-tuning ({EPOCHS} epochs)...\n")

for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*60}")
    
    # Training
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, loss_function, device
    )
    
    # Validation
    val_loss, val_acc, val_f1, val_preds, val_labels = eval_epoch(
        model, val_loader, loss_function, device
    )
    
    # Sauvegarde des métriques
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    # Affichage
    print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    
    # Sauvegarder le meilleur modèle (basé sur F1)
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch + 1
        patience_counter = 0
        
        # Sauvegarder le checkpoint
        checkpoint_path = os.path.join(checkpoint_dir, f"best_model.pt")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1': best_f1,
            'config': config,
        }, checkpoint_path)
        
        print(f"💾 Meilleur modèle sauvegardé (F1: {best_f1:.4f})")
    
    else:
        patience_counter += 1
        print(f"⏳ Pas d'amélioration ({patience_counter}/{patience})")
        
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping après {epoch + 1} epochs")
            break

print(f"\n{'='*60}")
print(f"✅ Entraînement terminé!")
print(f"Meilleur modèle à l'epoch {best_epoch} avec F1: {best_f1:.4f}")
print(f"{'='*60}")

In [ ]:
# 📊 Évaluation sur le Test Set
print(f"\n\n📊 Évaluation sur le Test Set\n")

# Charger le meilleur modèle
checkpoint_path = os.path.join(checkpoint_dir, "best_model.pt")
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Meilleur modèle chargé (F1: {checkpoint['best_f1']:.4f})")

# Évaluer
test_loss, test_acc, test_f1, test_preds, test_labels = eval_epoch(
    model, test_loader, loss_function, device
)

print(f"\n{'='*60}")
print(f"TEST SET RESULTS")
print(f"{'='*60}")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_acc:.4f}")
print(f"F1 Score (weighted): {test_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=list(REVERSE_EMOTION_MAP.values())))

# Matrice de confusion
cm = confusion_matrix(test_labels, test_preds)
print(f"\nConfusion Matrix:")
print(cm)

In [ ]:
# 📈 Visualisations
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss over Epochs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy over Epochs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['val_f1'], label='Val F1 Score', marker='o', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].set_title('F1 Score over Epochs')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/training_metrics.png", dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphes sauvegardés dans results/training_metrics.png")

In [ ]:
# 🔥 Heatmap Confusion Matrix
emotion_labels = [REVERSE_EMOTION_MAP[i] for i in range(NUM_CLASSES)]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=emotion_labels,
            yticklabels=emotion_labels, 
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix - Test Set (F1: {test_f1:.4f})', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig("../results/confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

print("✅ Matrice de confusion sauvegardée dans results/confusion_matrix.png")

In [ ]:
# 🎯 Fonction d'inférence
def predict_emotion(text, model, tokenizer, device):
    """
    Prédire l'émotion pour un texte donné.
    
    Args:
        text: str - Le texte à classifier
        model: modèle BERT
        tokenizer: BertTokenizer
        device: torch device
        
    Returns:
        emotion: str - L'émotion prédite
        confidence: float - La confiance de la prédiction
        probabilities: dict - Probabilités pour chaque émotion
    """
    model.eval()
    
    # Tokenize
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Forward pass
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
    
    # Get probabilities
    probs = torch.softmax(logits, dim=1)
    confidence, pred_idx = torch.max(probs, dim=1)
    
    emotion_idx = pred_idx.item()
    emotion = REVERSE_EMOTION_MAP[emotion_idx]
    confidence = confidence.item()
    
    # Probabilities dict
    probabilities = {
        REVERSE_EMOTION_MAP[i]: probs[0, i].item()
        for i in range(NUM_CLASSES)
    }
    
    return emotion, confidence, probabilities


# 🧪 Tester avec quelques exemples
test_sentences = [
    "I am so happy!",
    "This is terrible and makes me sad",
    "I am very angry at this situation",
    "I'm feeling calm and peaceful"
]

print("\n🧪 Test d'inférence:\n")

for sentence in test_sentences:
    emotion, confidence, probs = predict_emotion(sentence, model, tokenizer, device)
    print(f"Text: '{sentence}'")
    print(f"Prédiction: {emotion.upper()} (confiance: {confidence:.2%})")
    # Trier par probabilité
    top_3 = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"  Top-3: {top_3}")
    print()

In [ ]:
# 💾 Sauvegarder les artefacts finaux
print("\n💾 Sauvegarde des artefacts finaux...\n")

# 1. Modèle tokenizer
tokenizer.save_pretrained(checkpoint_dir)
print(f"✅ Tokenizer sauvegardé dans {checkpoint_dir}")

# 2. Fichier de configuration avec résultats
final_config = {
    **config,
    'results': {
        'train_loss': float(history['train_loss'][-1]),
        'train_acc': float(history['train_acc'][-1]),
        'val_loss': float(history['val_loss'][-1]) ,
        'val_acc': float(history['val_acc'][-1]),
        'val_f1': float(history['val_f1'][-1]),
        'test_loss': float(test_loss),
        'test_acc': float(test_acc),
        'test_f1': float(test_f1),
        'best_epoch': best_epoch,
        'emotion_map': EMOTION_MAP
    }
}

import json
config_save_path = os.path.join(checkpoint_dir, "config.json")
with open(config_save_path, 'w') as f:
    json.dump(final_config, f, indent=2)

print(f"✅ Configuration sauvegardée dans {config_save_path}")

# 3. Résumé
print(f"\n{'='*60}")
print(f"🎉 RÉSUMÉ FINAL")
print(f"{'='*60}")
print(f"Checkpoint: {checkpoint_dir}")
print(f"\nMétriques d'entraînement:")
print(f"  - Train Acc: {history['train_acc'][-1]:.4f}")
print(f"  - Val Acc: {history['val_acc'][-1]:.4f}")
print(f"  - Test Acc: {test_acc:.4f}")
print(f"\nMétriques de test:")
print(f"  - Test F1 (weighted): {test_f1:.4f}")
print(f"  - Test Loss: {test_loss:.4f}")
print(f"\nFichiers sauvegardés:")
print(f"  - Model weights: {checkpoint_dir}/best_model.pt")
print(f"  - Tokenizer: {checkpoint_dir}/")
print(f"  - Config: {config_save_path}")
print(f"  - Plots: ../results/")
print(f"{'='*60}")

## 📌 Utilisation quand la transcription sera prête

Une fois que ta binôme aura terminé la transcription avec Whisper, tu devras:

### 1. **Charger les vraies données**
Remplacer la section de chargement des données (cellule 3) avec:

```python
transcription_path = "../data/processed/transcriptions/transcriptions.csv"
df = pd.read_csv(transcription_path)  # Doit avoir colonnes: 'text', 'emotion'
```

### 2. **Adapter le format des données**
S'assurer que le DataFrame a les colonnes:
- `text` : La transcription texte
- `emotion` : L'étiquette d'émotion ('neutral', 'happy', 'sad', etc.)

### 3. **Exécuter le notebook**
Les cellules fonctionneront directement 🚀

---

## 🔑 Points clés à retenir

✅ **Statistical Pooling** : Moyenne des embeddings de tous les tokens (mieux que `[CLS]` seul)

✅ **Masking** : On applique un masque d'attention pour ignorer les tokens de padding

✅ **Fine-tuning** : On réentraîne le modèle BERT avec un learning rate bas (2e-5)

✅ **Early Stopping** : Le modèle s'arrête automatiquement après 3 epochs sans amélioration